# SIFESS — Kaggle / Colab training notebook

Day-0: **ChestMNIST** (MedMNIST) with DINO + soft isophote-frame equivariance.

Full NIH CXR: attach the dataset separately; see `docs/EXPERIMENT_CARD.md`.

**No fabricated metrics** — cells print whatever the run produces.


In [ ]:
# If running on Kaggle: enable GPU. Install from repo root or upload package.
import sys, os
from pathlib import Path

# Detect repo root (local clone or uploaded dataset)
CANDIDATES = [
    Path.cwd(),
    Path.cwd() / "sifess-medical-cv",
    Path("/kaggle/working/sifess-medical-cv"),
    Path("/kaggle/input/sifess-medical-cv"),
]
ROOT = None
for p in CANDIDATES:
    if (p / "pyproject.toml").exists() and (p / "sifess").is_dir():
        ROOT = p
        break
if ROOT is None:
    raise FileNotFoundError("Upload/clone sifess-medical-cv so pyproject.toml is visible")
os.chdir(ROOT)
print("ROOT", ROOT)
!pip -q install -e ".[dev]"


In [ ]:
import torch, sifess
from sifess.utils import set_seed
print("sifess", sifess.__version__)
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
set_seed(42)


In [ ]:
# Smoke / Day-0 SSL train (ResNet-18, tiny subset). Scale up epochs/subset for real runs.
from sifess.engines.train_ssl import train

cfg = dict(
    seed=42,
    ablation="sifess_full",
    backbone="resnet18",
    epochs=2,
    batch_size=16 if torch.cuda.is_available() else 8,
    subset=256,
    image_size=224,
    n_global=2,
    n_local=4,
    out_dim=1024,  # smaller for notebook speed
    lambda_eq=0.5,
    lambda_orth=0.01,
    sigma_rho=1.5,
    fp16=torch.cuda.is_available(),
    num_workers=2,
    lr=5e-4,
    weight_decay=0.04,
    teacher_momentum=0.996,
    output_dir="outputs/kaggle_ssl",
)
summary = train(cfg)
summary


In [ ]:
# Linear probe
from sifess.engines.linear_probe import run_probe

probe_summary = run_probe(dict(
    seed=42,
    checkpoint=summary["checkpoint"],
    backbone="resnet18",
    epochs=5,
    batch_size=32,
    subset=512,
    image_size=224,
    fp16=torch.cuda.is_available(),
    num_workers=2,
    lr=1e-3,
    output_dir="outputs/kaggle_probe",
))
probe_summary


In [ ]:
# OOD intensity eval
from sifess.engines.eval_ood import run_ood

ood_summary = run_ood(dict(
    seed=42,
    checkpoint=summary["checkpoint"],
    probe="outputs/kaggle_probe/probe.pt",
    backbone="resnet18",
    batch_size=32,
    subset=512,
    severity=0.75,
    image_size=224,
    num_workers=2,
    n_classes=14,
    output_dir="outputs/kaggle_ood",
))
ood_summary


## NIH ChestX-ray14 (full runs)

1. Add the NIH / ChestX-ray14 Kaggle dataset to the notebook.
2. Point `root` at the folder containing `Data_Entry_2017.csv`.
3. Use `sifess.data.chestxray_loader.build_nih_cxr` — **not wired into train_ssl yet** as Day-0 uses ChestMNIST.
4. Prefer ResNet-50, batch 64, fp16, seed 42; expect longer wall-clock.

Known gaps: official split CSVs must be provided; multi-label head size 14; no auto-download.


In [ ]:
# Optional: verify structure tensor on a ChestMNIST batch
from sifess.data.medmnist_loader import build_chestmnist
from sifess.geometry.structure_tensor import compute_structure_tensor

_, loader = build_chestmnist(split="train", batch_size=4, multicrop=False, subset=4)
batch = next(iter(loader))
st = compute_structure_tensor(batch["image"], sigma_rho=1.5)
print("Cbar", st.Cbar)
print("e1", st.e1.shape)
